<a href="https://colab.research.google.com/github/Andrei-WongE/advanced_geospatial_methods/blob/origin/GeoNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

4. Example on real data from Authors, in google colab

---



In [ ]:
# FIXED: Complete geospaNN Example_realdata
# ISSUE A: geospaNN/utils.py line ~96 passes verbose=True to ReduceLROnPlateau(), unsupported in Colab's PyTorch 2.4+.
# RESOLTUTION: Fix 1: Downgrade torch, DID NOT WORK; Fix 2: patching geospaNN based on error
# ISSUE B: The ordering of inputs x (covariates) and y (response) in BRISC_estimation has been changed BRISC 1.0.0 onwards.
# RESOLTUTION: expects BRISC <1.0.0

import torch  # ← FIRST!
import torch.optim.lr_scheduler as sched
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

# Patch class ReduceLROnPlateau ← THEN
class FixedReduceLR(sched.ReduceLROnPlateau):
    def __init__(self, *args, verbose=False, **kwargs):
        kwargs.pop('verbose', None)  # Remove verbose safely
        super().__init__(*args, **kwargs)

sched.ReduceLROnPlateau = FixedReduceLR

!pip install geospaNN geopandas[all] scipy seaborn matplotlib shapely rpy2==3.5.14 -q
!apt update -qq && apt install -y wget unzip -qq  # For data

import time
import os
import torch
import geospaNN
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from scipy import spatial, interpolate
import matplotlib.pyplot as plt

# Dont forget to change runtime to T4 GPU (this data set is small)
# print(f"CUDA available: {torch.cuda.is_available()}")
# print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
# if torch.cuda.is_available():
#     print(f"GPU: {torch.cuda.get_device_name(0)}")

# Download data
os.makedirs('./data', exist_ok=True)
!wget -q -O data/covariate0605.csv https://raw.githubusercontent.com/WentaoZhan1998/geospaNN/main/data/covariate0605.csv
!wget -q -O data/pm25_0605.csv https://raw.githubusercontent.com/WentaoZhan1998/geospaNN/main/data/pm25_0605.csv
!wget -q -O data/Normalized_PM2.5_20190605.csv https://raw.githubusercontent.com/WentaoZhan1998/geospaNN/main/data/Normalized_PM2.5_20190605.csv

print("Starting geospaNN Example_realdata...")

# Load US boundaries, need MultiPolygons as Hawaii must have their indiv rows BUT excludes Alaska, facilitates union later
url = "https://www2.census.gov/geo/tiger/GENZ2018/shp/cb_2018_us_nation_20m.zip"
us = gpd.read_file(url).explode()
us = us.loc[us.geometry.apply(lambda x: x.exterior.bounds[2]) < -60]

# Load data, excluding Northern Canada, US monitoring stations (~30-45°N)
df_covariates = pd.read_csv('./data/covariate0605.csv')
df_pm25 = pd.read_csv('./data/pm25_0605.csv')
df_pm25 = df_pm25.loc[df_pm25.Latitude < 50]

# Grid setup: covariate data bounds → 100×100 grid → 10,201 points → GeoDataFrame → Clip to USA
x_min, y_min, x_max, y_max = np.array([np.min(df_covariates['long']), np.min(df_covariates['lat']),
                                      np.max(df_covariates['long']), np.max(df_covariates['lat'])])
arr1 = np.mgrid[x_min:x_max:101j, y_min:y_max:101j]
## extract the x and y coordinates as flat arrays
arr1x = np.ravel(arr1[0])
arr1y = np.ravel(arr1[1])
## using the X and Y columns, build a dataframe, then the geodataframe
df = pd.DataFrame({'X': arr1x, 'Y': arr1y})
df['coords'] = list(zip(df['X'], df['Y']))
df['coords'] = df['coords'].apply(Point)

gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(x=df.X, y=df.Y), crs=us.crs)
inUS = gdf['geometry'].apply(lambda s: s.within(us.geometry.unary_union))

# Interpolate PM2.5 creating an continous surface
lonlat_pm25 = df_pm25.values[:, [1, 2]]
near = df_covariates.values[:, [1, 2]]
tree = spatial.KDTree(list(zip(near[:, 0].ravel(), near[:, 1].ravel())))
idx = tree.query(lonlat_pm25)[1]  # Find nearest covariate for each station
df_pm25_mean = df_pm25.assign(neighbor=idx).groupby('neighbor')['PM25'].mean() # Spatial averaging
idx_new = df_pm25_mean.index.values
pm25 = df_pm25_mean.values
z = pm25[:, None]

lon = df_covariates.values[:, 1]
lat = df_covariates.values[:, 2]

# Radial basis function (inverse multiquadric): Smoothly interpolates PM2.5 across grid
f = interpolate.Rbf(lon[idx_new], lat[idx_new], z, function='inverse')
x_test = gdf.loc[inUS, :].X
y_test = gdf.loc[inUS, :].Y
z_test = f(x_test, y_test)

# Plot 1: Interpolated PM2.5
plt.clf()
fig, ax = plt.subplots(figsize=(9, 5))
c = ax.scatter(x = x_test, y = y_test, s = 10, c = z_test, marker = 's', alpha = 0.7)
ax.plot(np.array(df_pm25['Longitude']), np.array(df_pm25['Latitude']), 'o', c = 'orange', markersize = 4)
ax.set_title('')
fig.colorbar(c, ax=ax)
plt.show()

# Normalization
lon = df_covariates.values[:,1]
lat = df_covariates.values[:,2]
covariates = df_covariates.values[:,3:]
normalized_lon = (lon-min(lon))/(max(lon)-min(lon))
normalized_lat = (lat-min(lat))/(max(lat)-min(lat))
normalized_x_test = (x_test-min(lon))/(max(lon)-min(lon))
normalized_y_test = (y_test-min(lat))/(max(lat)-min(lat))

s_obs = np.vstack((normalized_lon[idx_new],normalized_lat[idx_new])).T
X = covariates[idx_new,:]
normalized_X = X
for i in range(X.shape[1]):
    normalized_X[:,i] = (X[:,i]-min(X[:,i]))/(max(X[:,i])-min(X[:,i]))

X = normalized_X
Y = z.reshape(-1)
coord = s_obs
#columns = ['precipitation', 'temperature', 'air pressure', 'relative humidity', 'U-wind', 'V-wind',
#           'PM 2.5', 'longitude', 'latitude']
#df = pd.DataFrame(data=data, index=range(data.shape[0]), columns=columns)
#df.to_csv('./data/Normalized_PM2.5_20190605.csv')

# Load normalized data
data_PM25 = pd.read_csv("./data/Normalized_PM2.5_20190605.csv")
data_PM25

X = torch.from_numpy(data_PM25[['precipitation', 'temperature', 'air pressure', 'relative humidity', 'U-wind', 'V-wind']].to_numpy()).float()
Y = torch.from_numpy(data_PM25[['PM 2.5']].to_numpy().reshape(-1)).float()
coord = torch.from_numpy(data_PM25[['longitude', 'latitude']].to_numpy()).float()

p = X.shape[1]
n = X.shape[0]
nn = 20
batch_size = 50

X, Y, coord, _ = geospaNN.spatial_order(X, Y, coord, method='max-min')
data = geospaNN.make_graph(X, Y, coord, nn)

torch.manual_seed(2024)
torch.backends.cudnn.deterministic = True
np.random.seed(0)
data_train, data_val, data_test = geospaNN.split_data(X, Y, coord, neighbor_size=20, test_proportion=0.5)

# Train NN
# Input: p=6 covariates (precip, temp, pressure, humidity, U-wind, V-wind)
# Layer 1: 6 → 50 neurons + ReLU
# Layer 2: 50 → 20 neurons + ReLU
# Layer 3: 20 → 1 output (PM2.5 prediction)

print("Training NN...")
start_time = time.time()
mlp_nn = torch.nn.Sequential(
    torch.nn.Linear(p, 50),
    torch.nn.ReLU(),
    torch.nn.Linear(50, 20),
    torch.nn.ReLU(),
    torch.nn.Linear(20, 1),
)
# geospaNN.nn_train() wrapper creates trainer with:
#  Adam optimizer (lr=0.01)
#  ReduceLROnPlateau scheduler (halves lr on plateau)
#  EarlyStopping (stops if val loss doesn't improve by min_delta=0.001)
#  Handles PyTorch graph data automatically.
nn_model = geospaNN.nn_train(mlp_nn, lr=0.01, min_delta=0.001)
nn_model.train(data_train, data_val, data_test)

# RESULT: NNGLS = NN (non-spatial baseline + predictions/residual to initialise NNGLS) + Spatial Structure (using theta0)

# NN:        f(x) = MLP(x)
# NNGLS:     y = f(x) + GP(0, C(theta, pos))
# Where C = Matérn covariance via NNGP approximation

# Train NNGLS, using NN residuals that reveals spatial patterns → spatial correlation estimate → NNGLS converges faster
# Fits Matérn covariance to spatial residuals from MLP → initial spatial structure.
# More layers, NNGLS: 6→100→50→20→10→1 (5 layers)
print("Training NNGLS...")
# start_time = time.time()
theta0 = geospaNN.theta_update(torch.tensor([1, 1.5, 0.01]), mlp_nn(data_train.x).squeeze() - data_train.y, data_train.pos, neighbor_size=20)
mlp_nngls = torch.nn.Sequential(
    torch.nn.Linear(p, 100), torch.nn.ReLU(),
    torch.nn.Linear(100, 50), torch.nn.ReLU(),
    torch.nn.Linear(50, 20), torch.nn.ReLU(),
    torch.nn.Linear(20, 10), torch.nn.ReLU(),
    torch.nn.Linear(10, 1)
)
# geospaNN.nngls core components:
# - mlp_nngls:  Mean function (nonlinear)
# - theta0:    Spatial covariance parameters
# - neighbor_size=20: NNGP approximation (scales to millions of points)
model = geospaNN.nngls(p=p, neighbor_size=nn, coord_dimensions=2, mlp=mlp_nngls, theta=torch.tensor(theta0))
# nngls_train(): Specialized trainer alternating optimization:
#     Neural network weights (Adam)
#     Spatial parameters theta (updated every Update_step=10 epochs)
#     Early stopping + LR scheduling
nngls_model = geospaNN.nngls_train(model, lr=0.01, min_delta=0.001)
training_log = nngls_model.train(data_train, data_val, data_test, Update_init=20, Update_step=10)
end_time = time.time()
print(f"NNGLS complete in {end_time - start_time:.2f}s")
# RESULT: model predicts PM2.5 + uncertainty accounting for spatial correlation

# Predict & Plot 2: Truth vs Prediction
[test_predict, test_U, test_L] = model.predict(data_train, data_test, CI = True)

plt.clf()
plt.scatter(test_predict.detach().numpy(), data_test.y.detach().numpy(), s = 1, label = 'Truth vs prediction')
plt.scatter(data_test.y.detach().numpy(), data_test.y.detach().numpy(), s = 1, label = 'reference')
plt.xlabel("Prediction")
plt.ylabel("Truth")
plt.legend()
plt.show()

# Plot 3: Maps
f_pred = interpolate.CloughTocher2DInterpolator(list(zip(data_test.pos.detach().numpy()[:, 0],
                                                         data_test.pos.detach().numpy()[:, 1])),
                                                test_predict.detach().numpy())
f_true = interpolate.CloughTocher2DInterpolator(list(zip(data_test.pos.detach().numpy()[:, 0],
                                                         data_test.pos.detach().numpy()[:, 1])),
                                                data_test.y.detach().numpy())
f_L = interpolate.CloughTocher2DInterpolator(list(zip(data_test.pos.detach().numpy()[:, 0],
                                                      data_test.pos.detach().numpy()[:, 1])),
                                             test_L.detach().numpy())
f_U = interpolate.CloughTocher2DInterpolator(list(zip(data_test.pos.detach().numpy()[:, 0],
                                                      data_test.pos.detach().numpy()[:, 1])),
                                             test_U.detach().numpy())

titles = [['Prediction', 'Truth'], ['Lower CI', 'Upper CI']]
fig, ax = plt.subplots(2, 2, figsize=(16, 12))
for i in range(2):
    for j in range(2):
        if i == 0 and j == 0:
            im = ax[i,j].scatter(normalized_x_test, normalized_y_test, s=9, c=f_pred(normalized_x_test, normalized_y_test),
                                 marker='s', alpha=0.7, vmin=0, vmax=20)
        elif i == 0 and j == 1:
            im = ax[i,j].scatter(normalized_x_test, normalized_y_test, s=9, c=f_true(normalized_x_test, normalized_y_test),
                                 marker='s', alpha=0.7, vmin=0, vmax=20)
            ax[i,j].plot(data_test.pos.detach().numpy()[:,0], data_test.pos.detach().numpy()[:,1], 'o', c='orange', markersize=4)
        elif i == 1 and j == 0:
            im = ax[i,j].scatter(normalized_x_test, normalized_y_test, s=9, c=f_L(normalized_x_test, normalized_y_test),
                                 marker='s', alpha=0.7, vmin=0, vmax=20)
        else:
            im = ax[i,j].scatter(normalized_x_test, normalized_y_test, s=9, c=f_U(normalized_x_test, normalized_y_test),
                                 marker='s', alpha=0.7, vmin=0, vmax=20)
        ax[i,j].set_title(titles[i][j])
        plt.colorbar(im, ax=ax[i,j])
plt.tight_layout()
plt.show()

# PDP Plot
variable_names = ['Precipitation accumulation', 'Air temperature', 'Pressure', 'Relative humidity', 'U-wind', 'V-wind']
geospaNN.plot_PDP(model, X, variable_names)

print("Complete! All plots generated.")
data_PM25.head()


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 219.3/219.3 kB 3.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 7.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.0/210.0 kB 15.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.8/212.8 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.1/81.1 kB 6.6 MB/s eta 0:00:00
